# Notebook 05 — Evaluation & Ablation

**Goal**: Evaluate all pipelines on the held-out test set, compute all metrics, generate
comparative plots and LaTeX tables for the dissertation, and perform error/failure analysis.

| Pipeline | Classification | NLG | Retrieval |
|---|---|---|---|
| XLM-RoBERTa | ✓ | – | – |
| RemBERT | ✓ | – | – |
| Llama Zero-Shot | ✓ | ✓ | – |
| Llama CoT | ✓ | ✓ | – |
| Llama LoRA | ✓ | ✓ | – |
| Multi-Hop Agent | ✓ | ✓ | ✓ |

In [ ]:
# %pip install rouge-score bert-score sentence-transformers matplotlib pandas scikit-learn

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.metrics import (
    classification_metrics,
    nlg_metrics,
    retrieval_metrics,
    print_classification_metrics,
)

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR   = Path('../data/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LABEL_LIST = ['false', 'partially_true', 'true']
print('Imports ready.')

## 1. Load All Saved Results

In [ ]:
# Load gold test set
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv')
y_true  = test_df['veracity_label'].tolist()

# Load discriminative results
with open(PROCESSED_DIR / 'discriminative_results.json') as f:
    disc_results = json.load(f)

# Load generative results
with open(PROCESSED_DIR / 'generative_results.json') as f:
    gen_results = json.load(f)

# Load agent results (JSONL)
agent_df = pd.read_json(PROCESSED_DIR / 'agent_results.jsonl', lines=True)

# Load test set with LoRA predictions (for NLG eval)
lora_pred_df = pd.read_csv(PROCESSED_DIR / 'test_with_lora_preds.csv')

print(f'Loaded {len(test_df)} test rows.')
print(f'Agent records: {len(agent_df)}')

## 2. Classification Metrics — All Models

In [ ]:
rows = []

for model_key, label in [
    ('xlmr',    'XLM-RoBERTa'),
    ('rembert', 'RemBERT'),
]:
    m = disc_results[model_key]
    rows.append({
        'Model': label,
        'Accuracy': m['accuracy'],
        'Macro-F1': m['macro_f1'],
        **{f'F1-{k}': v for k, v in m['per_class_f1'].items()},
    })

for gen_key, label in [
    ('zero_shot',    'Llama Zero-Shot'),
    ('cot_baseline', 'Llama CoT'),
    ('lora_finetuned', 'Llama LoRA'),
]:
    m = gen_results[gen_key]
    rows.append({
        'Model': label,
        'Accuracy': m['accuracy'],
        'Macro-F1': m['macro_f1'],
        **{f'F1-{k}': v for k, v in m['per_class_f1'].items()},
    })

with open(PROCESSED_DIR / 'agent_metrics.json') as f:
    agent_metrics = json.load(f)

m = agent_metrics['classification']
rows.append({
    'Model': 'Multi-Hop Agent',
    'Accuracy': m['accuracy'],
    'Macro-F1': m['macro_f1'],
    **{f'F1-{k}': v for k, v in m['per_class_f1'].items()},
})

class_df = pd.DataFrame(rows)
display(class_df.set_index('Model').round(4))

## 3. NLG Metrics — Generative Models

In [ ]:
gold_justifications = test_df['justification'].fillna('').tolist()

nlg_rows = []

# LoRA predictions
lora_justs = lora_pred_df['just_lora'].fillna('').tolist()
lora_nlg = nlg_metrics(
    predictions=lora_justs,
    references=gold_justifications,
    include_bertscore=True,
    include_cosine=True,
)
nlg_rows.append({'Model': 'Llama LoRA', **lora_nlg})

# Agent justifications
agent_justs = agent_df['justification'].fillna('').tolist()
agent_nlg = nlg_metrics(
    predictions=agent_justs,
    references=gold_justifications[:len(agent_justs)],
    include_bertscore=True,
    include_cosine=True,
)
nlg_rows.append({'Model': 'Multi-Hop Agent', **agent_nlg})

nlg_df = pd.DataFrame(nlg_rows)
display(nlg_df.set_index('Model').round(4))

## 4. Retrieval Metrics — Agent Pipeline

In [ ]:
ret = agent_metrics['retrieval']
print('Retrieval Metrics:')
for k, v in ret.items():
    print(f'  {k}: {v:.4f}')

## 5. Comparative Bar Chart — Accuracy & Macro-F1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models = class_df['Model'].tolist()
x = np.arange(len(models))
width = 0.6

for ax, metric, color in zip(axes, ['Accuracy', 'Macro-F1'], ['steelblue', 'darkorange']):
    vals = class_df[metric].tolist()
    bars = ax.bar(x, vals, width, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha='right', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} — All Models')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'accuracy_macrof1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Per-Class F1 Heatmap

In [ ]:
f1_cols = [c for c in class_df.columns if c.startswith('F1-')]
heat_df = class_df.set_index('Model')[f1_cols]
heat_df.columns = [c.replace('F1-', '') for c in heat_df.columns]

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heat_df.values, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, fontsize=11)
ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=10)
plt.colorbar(im, ax=ax, label='F1 Score')

for i in range(len(heat_df.index)):
    for j in range(len(heat_df.columns)):
        val = heat_df.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='black' if val < 0.7 else 'white', fontsize=10)

ax.set_title('Per-Class F1 Score by Model')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'per_class_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. LaTeX Tables

In [ ]:
# Classification results table
latex_class = (
    class_df
    .set_index('Model')
    [['Accuracy', 'Macro-F1'] + f1_cols]
    .round(4)
    .to_latex(
        caption='Classification results on the held-out test set.',
        label='tab:classification_results',
        bold_rows=True,
        escape=False,
    )
)

with open(FIGURES_DIR / 'table_classification.tex', 'w', encoding='utf-8') as f:
    f.write(latex_class)

print('Classification LaTeX table saved.')
print(latex_class)

In [ ]:
# NLG results table
latex_nlg = (
    nlg_df
    .set_index('Model')
    .round(4)
    .to_latex(
        caption='NLG metrics for justification quality (generative models).',
        label='tab:nlg_results',
        bold_rows=True,
    )
)

with open(FIGURES_DIR / 'table_nlg.tex', 'w', encoding='utf-8') as f:
    f.write(latex_nlg)

print('NLG LaTeX table saved.')
print(latex_nlg)

## 8. Ablation — Agent vs. No-Decompose

In [ ]:
# Ablation: compare agent with decomposition vs. direct retrieval (no sub-questions)
# 'directly_retrieved' is defined as: run retriever on original claim only, no decompose step.
# We approximate this by checking if we stored single-hop retrieval in agent_df;
# if not, we note this as a future ablation run.

print('Ablation variants to report in dissertation:')
print('  A1: No decomposition (single-hop retrieval)')
print('  A2: BM25 only (no dense retrieval)')
print('  A3: Dense only (no BM25)')
print('  A4: Full multi-hop (BM25 + dense + decompose) ← main system')
print()
print('To run A1–A3 ablations, re-run Notebook 04 with the appropriate retriever')
print('and n_subquestions=0 override, then load results here for comparison.')

## 9. Error / Failure Analysis

In [ ]:
# Identify failure cases: agent predicted wrong
agent_df['correct'] = agent_df['pred_label'] == agent_df['true_label']
failure_df = agent_df[~agent_df['correct']].copy()

print(f'Total failures: {len(failure_df)} / {len(agent_df)} ({len(failure_df)/len(agent_df)*100:.1f}%)')
print()
print('Failure distribution by true label:')
print(failure_df['true_label'].value_counts())
print()
print('Confusion (true_label → pred_label):')
print(failure_df.groupby(['true_label', 'pred_label']).size().unstack(fill_value=0))

In [ ]:
# Save failure cases
failure_cols = ['claim', 'true_label', 'pred_label', 'justification', 'sub_questions', 'evidence_str']
avail_cols = [c for c in failure_cols if c in failure_df.columns]
failure_df[avail_cols].to_csv(
    PROCESSED_DIR / 'failure_cases.csv', index=False, encoding='utf-8'
)
print('Failure cases saved to data/processed/failure_cases.csv')

In [ ]:
# Show the 5 most interesting failures (confident wrong predictions)
sample_failures = failure_df[avail_cols].head(5)
for _, row in sample_failures.iterrows():
    print('CLAIM   :', str(row.get('claim', ''))[:120])
    print('TRUE    :', row.get('true_label', ''))
    print('PRED    :', row.get('pred_label', ''))
    print('JUSTIF  :', str(row.get('justification', ''))[:200])
    print('-' * 80)

## 10. Final Summary Table

In [ ]:
print('=' * 70)
print('FINAL RESULTS SUMMARY')
print('=' * 70)
display(class_df.set_index('Model')[['Accuracy', 'Macro-F1']].round(4))
print()
print('NLG Metrics (justification quality):')
display(nlg_df.set_index('Model').round(4))
print()
print('Retrieval Metrics (Multi-Hop Agent):')
for k, v in ret.items():
    print(f'  {k}: {v:.4f}')